# Test: MCMC Data Augmentation

Tests for `src/optim/mcmc_augment.py`:
- `chord_interval`  — analytical chord intersection with Polytope #1
- `activation_pattern` — ReLU activation pattern extraction
- `find_augmented_point` — hit-and-run MCMC to find x' with different q-model pattern

Switch between MLP and CNN via the **Config** cell below.

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

# Project root = one level above this notebook
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, ROOT)   # project root already defined in Config cell

import numpy as np
import torch

from src.models.networks           import FashionMLP_Large, FashionCNN_Small
from src.quantization.quantize     import quantize_model
from src.optim.build_polytopes     import build_all_polytopes
from src.optim.build_polytopes_cnn import build_cnn_all_polytopes
from src.optim.mcmc_augment        import chord_interval, activation_pattern, find_augmented_point

## Config — switch MODEL_TYPE here

In [2]:
MODEL_TYPE  = "cnn"   # "mlp" or "cnn"
SAMPLE_IDX  = 0
BITS        = 4       # quantization bit-width used for the q-model
MAX_TRIES   = 200     # max MCMC attempts in find_augmented_point
SEED        = 42
rng = np.random.default_rng(SEED)


MODEL_PATHS = {
    "mlp": os.path.join(ROOT, "checkpoints/fashion_mlp_best.pth"),
    "cnn": os.path.join(ROOT, "checkpoints/fashion_cnn_best.pth"),
}
DATA_PATHS = {
    "mlp": os.path.join(ROOT, "data/fashionMNIST_correct_mlp.pt"),
    "cnn": os.path.join(ROOT, "data/fashionMNIST_correct_cnn.pt"),
}

print(f"Project root: {ROOT}")

Project root: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


## Load model, q-model, and sample

In [3]:
# --- Full-precision model ---
if MODEL_TYPE == "cnn":
    model = FashionCNN_Small()
else:
    model = FashionMLP_Large()

device = "mps" if torch.backends.mps.is_available() else "cpu"

state_dict = torch.load(MODEL_PATHS[MODEL_TYPE], 
                        weights_only=True, 
                        map_location=device)
model.load_state_dict(state_dict)
model.eval()
print(f"Loaded {MODEL_TYPE.upper()} model")

# --- Quantized model (single bit-width) ---
qmodel = quantize_model(model, bits=BITS)
qmodel.eval()
print(f"Quantized model: {BITS} bits")

# --- Sample ---
dataset = torch.load(DATA_PATHS[MODEL_TYPE], weights_only=False)
x_raw, c = dataset[SAMPLE_IDX]
c = int(c)
print(f"Sample {SAMPLE_IDX}: shape={x_raw.shape}  label={c}")

Loaded CNN model
Quantized model: 4 bits
Sample 0: shape=torch.Size([1, 28, 28])  label=9


## Build Polytope #1 (A_base, b_base)

In [4]:
# Prepare input batch and build polytopes — same convention as run_volumes.py
if MODEL_TYPE == "cnn":
    x_batch = x_raw.unsqueeze(0)             # (1, 1, 28, 28)
    x0      = x_raw.unsqueeze(0)             # same shape passed to find_augmented_point
    A_base, b_base, _ = build_cnn_all_polytopes(model, {BITS: qmodel}, x_batch, c)
else:
    x_batch = x_raw.flatten().unsqueeze(0)   # (1, 784)
    x0      = x_raw.flatten().unsqueeze(0)   # same shape passed to find_augmented_point
    A_base, b_base, _ = build_all_polytopes(model, {BITS: qmodel}, x_batch, c)

# Convert to numpy for LP / chord computations
A_np = A_base.detach().cpu().numpy()   # (m, 784)
b_np = b_base.detach().cpu().numpy()   # (m,)

print(f"Polytope #1:  A shape = {A_np.shape}")

# Sanity check: x0 must be feasible (all Ax+b <= 0)
x0_flat = x0.detach().cpu().numpy().flatten()
slack = A_np @ x0_flat + b_np
print(f"Max constraint violation at x0: {slack.max():.2e}  (should be <= 0 up to some numerical tolerance)")

Polytope #1:  A shape = (20685, 784)
Max constraint violation at x0: 1.19e-07  (should be <= 0 up to some numerical tolerance)


## Test 1 — `chord_interval`

In [5]:
n_test_directions = 10
print(f"Testing chord_interval on {n_test_directions} random directions...\n")

# chord_interval adds +1e-6 to s internally, so endpoints have slack ≈ 1e-6
# by construction.  Floating-point in the large matmul adds ~1e-7, so we use
# a slightly larger test tolerance (1e-5) to avoid spurious failures.
eps_test = 1e-5

all_ok = True
for i in range(n_test_directions):
    d = rng.standard_normal(x0_flat.shape[0])
    d /= np.linalg.norm(d)

    t_min, t_max = chord_interval(x0_flat, A_np, b_np, d)

    # 1. t_min < 0 < t_max  (x0 is strictly interior)
    sign_ok = t_min < 0 < t_max

    # 2. Both endpoints satisfy Ax+b <= 0  (up to eps_test tolerance)
    x_lo = x0_flat + t_min * d
    x_hi = x0_flat + t_max * d
    lo_ok = (A_np @ x_lo + b_np).max() <= eps_test
    hi_ok = (A_np @ x_hi + b_np).max() <= eps_test

    status = "OK" if (sign_ok and lo_ok and hi_ok) else "FAIL"
    if status == "FAIL":
        all_ok = False

    print(f"  dir {i:2d}: t_min={t_min:+.4f}  t_max={t_max:+.4f}  "
          f"sign={'ok' if sign_ok else 'FAIL'}  "
          f"endpoints={'ok' if (lo_ok and hi_ok) else 'FAIL'}  [{status}]")

print(f"\nAll tests passed: {all_ok}")

Testing chord_interval on 10 random directions...



  dir  0: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]
  dir  1: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]
  dir  2: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]
  dir  3: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]
  dir  4: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]
  dir  5: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]
  dir  6: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]
  dir  7: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]
  dir  8: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]
  dir  9: t_min=-0.0000  t_max=+0.0000  sign=ok  endpoints=ok  [OK]

All tests passed: True


## Test 2 — `activation_pattern`

In [6]:
# Pattern at x0 for full-precision model and q-model
pat_model  = activation_pattern(x0, model)
pat_qmodel = activation_pattern(x0, qmodel)

n_neurons = len(pat_model)
n_active_model  = pat_model.sum()
n_active_qmodel = pat_qmodel.sum()
n_differ = (pat_model != pat_qmodel).sum()

print(f"Total ReLU neurons\t: {n_neurons}")
print(f"Active in model  (fp)\t: {n_active_model}  ({100*n_active_model/n_neurons:.1f}%)")
print(f"Active in qmodel ({BITS}b)\t: {n_active_qmodel}  ({100*n_active_qmodel/n_neurons:.1f}%)")
print(f"Neurons differing at x0\t: {n_differ}  ({100*n_differ/n_neurons:.1f}%)")
print()

# Determinism check: calling twice should give identical results
pat2 = activation_pattern(x0, qmodel)
print(f"Determinism check: patterns identical = {np.array_equal(pat_qmodel, pat2)}")

Total ReLU neurons	: 11856
Active in model  (fp)	: 3578  (30.2%)
Active in qmodel (4b)	: 3991  (33.7%)
Neurons differing at x0	: 829  (7.0%)

Determinism check: patterns identical = True


## Test 3 — `find_augmented_point`

In [7]:
import time

t0 = time.perf_counter()
x_prime = find_augmented_point(x0, A_np, b_np, qmodel, max_tries=MAX_TRIES, rng=rng)
elapsed = time.perf_counter() - t0

if x_prime is None:
    print(f"No augmented point found within {MAX_TRIES} tries.")
else:
    x_prime_flat = x_prime.detach().cpu().numpy().flatten()

    # --- Check 1: x' is inside Polytope #1 ---
    # Polytope #1 encodes both the model's activation pattern AND its
    # classification decision.  Membership guarantees both properties below.
    # Tolerance 1e-5: chord_interval adds 1e-6 slack internally; floating-point
    # in the matmul adds ~1e-7, so 1e-5 gives safe margin without being loose.
    slack_prime = A_np @ x_prime_flat + b_np
    inside = slack_prime.max() <= 1e-5
    print(f"[1] x' inside Polytope #1:              {inside}  (max slack = {slack_prime.max():.2e})")

    # --- Check 2: model predicts the same class c at x' ---
    # Theoretically guaranteed by Check 1 (classification constraints are
    # part of A_base), but verified explicitly as a numerical sanity check.
    with torch.no_grad():
        logits_prime = model(x_prime)
    c_pred = int(logits_prime.argmax(dim=-1).item())
    class_ok = (c_pred == c)
    print(f"[2] model(x') predicts class {c_pred} (expected {c}):  {'OK' if class_ok else 'FAIL'}")

    # --- Check 3: q-model activation pattern at x' differs from x0 ---
    pat_prime = activation_pattern(x_prime, qmodel)
    n_flipped = (pat_prime != pat_qmodel).sum()
    print(f"[3] Neurons with different q-model pattern:  {n_flipped} / {len(pat_prime)}")

    # --- Check 4: x' has the same shape as x0 ---
    print(f"[4] Shape of x':  {tuple(x_prime.shape)}  (expected {tuple(x0.shape)})")

    print(f"\nElapsed: {elapsed:.2f}s")

[1] x' inside Polytope #1:              True  (max slack = 9.54e-07)
[2] model(x') predicts class 9 (expected 9):  OK
[3] Neurons with different q-model pattern:  80 / 11856
[4] Shape of x':  (1, 1, 28, 28)  (expected (1, 1, 28, 28))

Elapsed: 0.02s


## Acceptance rate over multiple samples

In [8]:
# Run find_augmented_point on N_PROBE samples to estimate success rate and speed.
N_PROBE = 5

successes   = 0
class_fails = 0
times       = []

for idx in range(N_PROBE):
    x_i, c_i = dataset[idx]
    c_i = int(c_i)

    if MODEL_TYPE == "cnn":
        x_b    = x_i.unsqueeze(0)
        x_i_in = x_i.unsqueeze(0)
        A_i, b_i, _ = build_cnn_all_polytopes(model, {BITS: qmodel}, x_b, c_i)
    else:
        x_b    = x_i.flatten().unsqueeze(0)
        x_i_in = x_i.flatten().unsqueeze(0)
        A_i, b_i, _ = build_all_polytopes(model, {BITS: qmodel}, x_b, c_i)

    A_i_np = A_i.detach().cpu().numpy()
    b_i_np = b_i.detach().cpu().numpy()

    t0 = time.perf_counter()
    xp = find_augmented_point(x_i_in, A_i_np, b_i_np, qmodel, max_tries=MAX_TRIES, rng=rng)
    dt = time.perf_counter() - t0
    times.append(dt)

    if xp is None:
        print(f"  sample {idx}: NOT found   ({dt:.2f}s)")
        continue

    successes += 1

    # Check 1: x' inside Polytope #1  (tolerance 1e-5, same reasoning as Test 1)
    xp_flat = xp.detach().cpu().numpy().flatten()
    slack   = (A_i_np @ xp_flat + b_i_np).max()
    inside  = slack <= 1e-5

    # Check 2: model predicts same class c
    with torch.no_grad():
        c_pred = int(model(xp).argmax(dim=-1).item())
    class_ok = (c_pred == c_i)
    if not class_ok:
        class_fails += 1

    print(f"  sample {idx}: found  ({dt:.2f}s)  "
          f"inside={inside} (slack={slack:.1e})  "
          f"class={c_pred}={'OK' if class_ok else 'FAIL'}")

print(f"\nSuccess rate:      {successes}/{N_PROBE}  ({100*successes/N_PROBE:.0f}%)")
print(f"Class pred fails:  {class_fails}/{successes}")
print(f"Mean time/sample:  {np.mean(times):.2f}s")

  sample 0: found  (0.02s)  inside=True (slack=3.3e-07)  class=9=OK
  sample 1: found  (0.03s)  inside=True (slack=8.3e-07)  class=0=OK
  sample 2: found  (0.03s)  inside=True (slack=1.1e-06)  class=0=OK
  sample 3: found  (0.02s)  inside=True (slack=6.0e-07)  class=3=OK
  sample 4: found  (0.03s)  inside=True (slack=9.7e-07)  class=0=OK

Success rate:      5/5  (100%)
Class pred fails:  0/5
Mean time/sample:  0.03s
